In [ ]:
import re
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTROL_TABLE = "ctl_monthly_load"
BRONZE_TABLE = "bronze_ontime"
RAW_ROOT = "Files/raw/bts_ontime"


def period_folder(period_key: str) -> str:
    return f"{RAW_ROOT}/{period_key.replace('-', '_')}"


def find_csv_files(folder: str) -> list:
    if not notebookutils.fs.exists(folder):
        return []
    found = []
    for item in notebookutils.fs.ls(folder):
        if item.isDir:
            found.extend(find_csv_files(item.path))
        elif item.name.lower().endswith(".csv") and item.size > 0:
            found.append(item)
    return found


def to_snake_case(name: str) -> str:
    s = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name.strip())
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    s = re.sub(r"[^A-Za-z0-9]+", "_", s)
    return s.strip("_").lower()


def update_status(period_keys: list, column: str, value: str) -> None:
    if not period_keys:
        return
    updates = spark.createDataFrame([(k,) for k in period_keys], ["period_key"])
    (
        DeltaTable.forName(spark, CONTROL_TABLE).alias("t")
        .merge(updates.alias("u"), "t.period_key = u.period_key")
        .whenMatchedUpdate(set={
            column: F.lit(value),
            "last_updated": F.current_timestamp(),
        })
        .execute()
    )


print("Configuration loaded.")

Configuration loaded.


In [ ]:
pending = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("download_status") == "pending")
    .select("period_key")
    .orderBy("period_key")
    .collect()
)

newly_downloaded = [
    r.period_key
    for r in pending
    if find_csv_files(period_folder(r.period_key))
]

update_status(newly_downloaded, "download_status", "downloaded")

print(f"Months checked: {len(pending)}")
print(f"Marked as downloaded: {len(newly_downloaded)} {newly_downloaded}")

Months checked: 90
Marked as downloaded: 1 ['2026-06']


In [ ]:
to_load = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("download_status") == "downloaded")
        & (F.col("load_status") == "pending")
    )
    .select("period_key")
    .orderBy("period_key")
    .collect()
)

loaded, failed = [], []

for r in to_load:
    period_key = r.period_key
    try:
        csv_files = find_csv_files(period_folder(period_key))
        if not csv_files:
            raise FileNotFoundError(f"No CSV found for {period_key}")

        month_df = None
        for f in csv_files:
            df = (
                spark.read
                .option("header", True)
                .option("inferSchema", False)
                .csv(f.path)
            )

            empty_cols = [c for c in df.columns if re.fullmatch(r"_c\d+", c)]
            df = df.drop(*empty_cols)
            df = df.toDF(*[to_snake_case(c) for c in df.columns])

            df = (
                df
                .withColumn("_period_key", F.lit(period_key))
                .withColumn("_source_file", F.lit(f.name))
                .withColumn("_ingested_at", F.current_timestamp())
            )

            month_df = df if month_df is None else month_df.unionByName(df, allowMissingColumns=True)

        writer = (
            month_df.write
            .format("delta")
            .option("mergeSchema", "true")
            .partitionBy("_period_key")
        )

        if spark.catalog.tableExists(BRONZE_TABLE):
            writer = (
                writer
                .mode("overwrite")
                .option("replaceWhere", f"_period_key = '{period_key}'")
            )
        else:
            writer = writer.mode("overwrite")

        writer.saveAsTable(BRONZE_TABLE)

        row_count = spark.table(BRONZE_TABLE).filter(F.col("_period_key") == period_key).count()
        update_status([period_key], "load_status", "loaded")
        loaded.append(period_key)
        print(f"Loaded {period_key}: {row_count:,} rows")

    except Exception as e:
        failed.append(period_key)
        print(f"FAILED {period_key}: {str(e)[:300]}")

print(f"\nMonths loaded: {len(loaded)}")
print(f"Months failed: {len(failed)} {failed}")

if spark.catalog.tableExists(BRONZE_TABLE):
    bronze = spark.table(BRONZE_TABLE)
    print(f"Bronze columns (excluding audit): {len(bronze.columns) - 3}")
    print(f"First 8 columns: {bronze.columns[:8]}")

if failed:
    raise RuntimeError(f"{len(failed)} month(s) failed to load: {failed}")

Loaded 2026-06: 607,577 rows

Months loaded: 1
Months failed: 0 []
Bronze columns (excluding audit): 109
First 8 columns: ['year', 'quarter', 'month', 'dayof_month', 'day_of_week', 'flight_date', 'reporting_airline', 'dot_id_reporting_airline']


In [ ]:
months_loaded = len(loaded)
print(f"Exiting with months_loaded = {months_loaded}")
notebookutils.notebook.exit(str(months_loaded))